In [2]:
import soundfile as sf
import torch, torchaudio
from pydub import AudioSegment
import librosa
from IPython.display import display, Audio
import numpy as np

In [3]:
AUDIO_PATH = './sample.wav'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

In [6]:
def pitch_shift():
    pitch_rate = int(input("Enter pitch rate: "))
    audio, sr = librosa.load(AUDIO_PATH)

    audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=pitch_rate)
    sf.write('pitch_shift.wav', audio, sr)

In [27]:
def add_white_noise():
    white_noise_int = float(input("enter white noise: "))
    audio, sr = librosa.load(AUDIO_PATH, sr=None)

    white_noise = np.random.normal(0, audio.std(), len(audio))

    white_noise_audio = audio + white_noise * white_noise_int
    sf.write('white_noise2.wav', white_noise_audio, sr)

In [29]:
def repeat():
    repeat_int = 5
    audio = AudioSegment.from_wav('./ori.wav')

    repeat_audio = audio * repeat_int

    repeat_audio.export('repeated1.wav', format="wav")

In [31]:
def insert_fade_in():
    fade_in = 1000
    audio = AudioSegment.from_wav('./ori.wav')

    fade_audio = audio.fade_in(fade_in)

    fade_audio.export('fade_audio1.wav', format="wav")

In [32]:
# pitch_shift()
# add_white_noise()
# repeat()
insert_fade_in()

In [33]:
def predict_text():
    audio, sr = torchaudio.load(AUDIO_PATH)

    if sr != bundle.sample_rate:
        audio = torchaudio.functional.resample(audio, sr, bundle.sample_rate)

    with torch.inference_mode():
        emission, _ = model(audio)

    labels = bundle.get_labels()

    idx = torch.argmax(emission[0], dim=-1)
    idx = torch.unique_consecutive(idx, dim=-1)
    idx = [i for i in idx if i != 0]

    pred = "".join([labels[i] for i in idx])

    print(pred.replace("|", " "))

In [34]:
predict_text()

I REALLY LOVE TITE SO I LIKE TO HUG TETE 
